### Decoder Block
- Encoder-Decoder Attention을 제거 (Decoder-only 구조는 Encoder가 없으므로 Decoder내부에서만 (masked) self-attention만 사용)
- 따라서 Decoder Block은 Self-Attention과 Feed-Forward Network 두 개의 레이어로만 구성

### GPT(모델)
- Positional Encoding을 사용하지 않고, nn.Embedding을 사용(learnable한 position layer)

In [1]:
import os, re, io, math
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import sentencepiece as spm

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
def normalize_text(sentence):
    sentence = sentence.lower().strip()

    #단어와 구두점(punctuation) 사이의 거리를 만듦.
    #단어와 온점 사이에 거리를 만듦.
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)

    #(a-z, A-Z, 한글, 숫자, ".", "?", "!", ",")를 제외한 모든 문자를 공백인 ' '로 대체.
    sentence = re.sub(r"[^가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    sentence = sentence.strip()

    return sentence

CSV_PATH = os.path.join(DATA_DIR, "ChatbotData.csv")
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=["Q", "A"]).reset_index(drop=True)

df["Q"] = df["Q"].astype(str).map(normalize_text)
df["A"] = df["A"].astype(str).map(normalize_text)
df = df[(df["Q"] != "") & (df["A"] != "")].reset_index(drop=True)

pairs = list(zip(df["Q"].tolist(), df["A"].tolist()))
print(f"[데이터] 총 페어 수: {len(pairs):,}")

[데이터] 총 페어 수: 11,823


In [3]:
pairs[0:5]

[('12시 땡 !', '하루가 또 가네요 .'),
 ('1지망 학교 떨어졌어', '위로해 드립니다 .'),
 ('3박4일 놀러가고 싶다', '여행은 언제나 좋죠 .'),
 ('3박4일 정도 놀러가고 싶다', '여행은 언제나 좋죠 .'),
 ('ppl 심하네', '눈살이 찌푸려지죠 .')]

In [4]:
print("[SentencePiece] 학습 말뭉치 생성…")
corpus_path = os.path.join(DATA_DIR, "spm_corpus.txt")

with io.open(corpus_path, "w", encoding="utf-8") as f:
    for q, a in pairs:
        f.write(q + "\n")
        f.write(a + "\n")

[SentencePiece] 학습 말뭉치 생성…


In [5]:
SPM_PREFIX = os.path.join(DATA_DIR, "spm_kor")
VOCAB_SIZE = 8000

print("[SentencePiece] 학습 시작…")
spm.SentencePieceTrainer.Train(
    input=corpus_path,
    model_prefix=SPM_PREFIX,
    vocab_size=VOCAB_SIZE,
    model_type="unigram", # or bpe
    character_coverage=0.9995,
    pad_id=0,      # <pad>
    bos_id=1,      # <s>
    eos_id=2,      # </s>
    unk_id=3,      # <unk>
    user_defined_symbols=[]
)

sp = spm.SentencePieceProcessor()
sp.Load(SPM_PREFIX + ".model")
PAD_ID = sp.pad_id()     # 0
BOS_ID = sp.bos_id()     # 1
EOS_ID = sp.eos_id()     # 2
UNK_ID = sp.unk_id()     # 3
VOCAB_SIZE = sp.GetPieceSize()
print(f"[SPM] vocab size: {VOCAB_SIZE}, PAD/BOS/EOS/UNK={PAD_ID}/{BOS_ID}/{EOS_ID}/{UNK_ID}")

[SentencePiece] 학습 시작…
[SPM] vocab size: 8000, PAD/BOS/EOS/UNK=0/1/2/3


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./data/spm_corpus.txt
  input_format: 
  model_prefix: ./data/spm_kor
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_priv

In [6]:
sp = spm.SentencePieceProcessor()
sp.Load(SPM_PREFIX + ".model")
PAD_ID = sp.pad_id()     # 0
BOS_ID = sp.bos_id()     # 1
EOS_ID = sp.eos_id()     # 2
UNK_ID = sp.unk_id()     # 3
VOCAB_SIZE = sp.GetPieceSize()
print(f"[SPM] vocab size: {VOCAB_SIZE}, PAD/BOS/EOS/UNK={PAD_ID}/{BOS_ID}/{EOS_ID}/{UNK_ID}")

[SPM] vocab size: 8000, PAD/BOS/EOS/UNK=0/1/2/3


In [7]:
# 예제 문장
sentence = "안녕하세요. 제 이름은 최은학입니다."

sentence = normalize_text(sentence)
print("전처리 후의 문장:", sentence)

# 1. 토크나이징 (subword 단위로 분할)
tokens = sp.encode(sentence, out_type=str)
print("Tokenized:", tokens)

# 2. 인코딩 (서브워드를 정수 ID로 변환)
encoded = sp.encode(sentence, out_type=int)
print("Encoded:", encoded)

# 3. 디코딩 (정수 ID → 원본 문장 복원)
decoded = sp.decode(encoded)
print("Decoded:", decoded)

전처리 후의 문장: 안녕하세요 . 제 이름은 최은학입니다 .
Tokenized: ['▁안녕하세요', '▁.', '▁제', '▁이름', '은', '▁', '최', '은', '학', '입니다', '▁.']
Encoded: [3118, 4, 335, 3212, 15, 5, 7955, 15, 2085, 262, 4]
Decoded: 안녕하세요 . 제 이름은 최은학입니다 .


In [8]:
def create_padding_mask(x):
    # x == 0 위치를 찾아 float형 1로 변환
    mask = (x == 0).float()
    # (batch_size, seq_len) -> (batch_size, 1, 1, seq_len)
    mask = mask.unsqueeze(1).unsqueeze(2)
    return mask

def create_look_ahead_mask(x):
    seq_len = x.size(1)

    # (seq_len, seq_len) 크기의 하삼각 행렬(tril) 생성 후 1에서 빼서
    # 상삼각이 1, 하삼각(자기 자신 포함)이 0이 되도록 설정
    # => 미래 토큰(자신 인덱스보다 큰 위치) 마스킹
    look_ahead_mask = 1 - torch.tril(torch.ones((seq_len, seq_len)))

    # 패딩 마스크 생성 (shape: (batch_size, 1, 1, seq_len))
    padding_mask = create_padding_mask(x)

    # look_ahead_mask: (seq_len, seq_len) -> (1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(0)
    # -> (1, seq_len, seq_len) -> (1, 1, seq_len, seq_len)
    look_ahead_mask = look_ahead_mask.unsqueeze(1)
    look_ahead_mask = look_ahead_mask.to(x.device)

    # look-ahead 마스크와 패딩 마스크를 합성 (둘 중 하나라도 1이면 마스킹)
    # 최종 shape은 브로드캐스팅으로 (batch_size, 1, seq_len, seq_len)
    combined_mask = torch.max(look_ahead_mask, padding_mask)
    return combined_mask

In [9]:
class GPTDataset(Dataset):
    def __init__(self, pairs, sp, max_length=40):
        super().__init__()
        self.sp = sp
        self.max_length = max_length # 모델에 입력될 최종 시퀀스의 길이
        self.data = []

        # SentencePiece에서 정의된 특수 토큰 ID 가져오기
        bos_id = self.sp.bos_id() # 문장 시작 토큰
        eos_id = self.sp.eos_id() # 문장 끝 토큰
        pad_id = self.sp.pad_id() # 패딩 토큰

        print("[데이터셋] Q, A 쌍을 단일 시퀀스로 변환")
        for q_text, a_text in pairs:
            # 1. 질문과 답변을 각각 토큰 ID로 변환
            q_ids = self.sp.EncodeAsIds(q_text)
            a_ids = self.sp.EncodeAsIds(a_text)

            # 2. Q와 A를 하나의 연속된 시퀀스로 결합
            # 형식: [BOS] Q tokens [EOS] A tokens [EOS]
            # 질문과 답변 사이에 구분자 역할로 EOS 토큰을 사용
            combined_ids = [bos_id] + q_ids + [eos_id] + a_ids + [eos_id]

            # 3. 최대 길이를 초과하는 시퀀스는 데이터셋에서 제외
            if len(combined_ids) > self.max_length:
                continue

            # 4. 패딩(Padding) 추가
            # 모든 시퀀스의 길이를 max_length로 통일
            padding_len = self.max_length - len(combined_ids)
            combined_ids += [pad_id] * padding_len

            # 5. 입력(input)과 정답(label) 생성
            # 언어 모델링을 위해, 시퀀스를 한 칸씩 shift해 입/출력 쌍을 만듦
            # 예: [1, 4, 8, 5, 2] -> input: [1, 4, 8, 5], label: [4, 8, 5, 2]
            # look_ahead_mask를 적용하게 되면, 이전 시퀀스를 통해 다음 토큰을 예측하도록 학습하게 됨.
            # 아에 모든 데이터의 time step에 대해서 이전 시퀀스를 input으로, 다음 토큰을 target으로 두고 mask를 사용하지 않아도 되는데
            # 이렇게하면 효율성이 떨어짐. (데이터 수가 각 length배 만큼 증가)
            model_input = combined_ids[:-1]
            target_label = combined_ids[1:]

            self.data.append({
                "input": model_input,
                "label": target_label
            })
        print(f"[데이터셋] 총 {len(self.data):,}개의 유효한 시퀀스 생성")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        model_input = torch.tensor(sample["input"], dtype=torch.long)
        target_label = torch.tensor(sample["label"], dtype=torch.long)

        # 이제 인코더 입력 없이 (모델 입력, 정답 레이블) 두 가지만 반환
        return model_input, target_label



dataset = GPTDataset(pairs, sp, max_length=40)

dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

for model_input, target_label in dataloader:
    print("모델 입력의 배치 크기:", model_input.size())
    print("정답 레이블의 배치 크기:", target_label.size())
    print("-" * 30)

    # 첫 번째 샘플 출력
    sample_input = model_input[0]
    sample_label = target_label[0]

    print("샘플 입력 (ID):\n", sample_input)
    print("샘플 입력 (복원):\n", sp.decode(sample_input.tolist()))
    print("-" * 30)
    print("샘플 정답 (ID):\n", sample_label)
    print("샘플 정답 (복원):\n", sp.decode(sample_label.tolist()))
    break

[데이터셋] Q, A 쌍을 단일 시퀀스로 변환 중…
[데이터셋] 총 11,820개의 유효한 시퀀스 생성 완료.
11820
모델 입력의 배치 크기: torch.Size([64, 39])
정답 레이블의 배치 크기: torch.Size([64, 39])
------------------------------
샘플 입력 (ID):
 tensor([   1, 2457,   29,   27,  194,   12,  780,  770,    7,    2,  123,   15,
         934,  205,  142,   14,    4,    2,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])
샘플 입력 (복원):
 노는 거 좋아하는 남자는 좋아하면 힘들겠지 ? 지금은 괜찮지만 힘들 거예요 .
------------------------------
샘플 정답 (ID):
 tensor([2457,   29,   27,  194,   12,  780,  770,    7,    2,  123,   15,  934,
         205,  142,   14,    4,    2,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0])
샘플 정답 (복원):
 노는 거 좋아하는 남자는 좋아하면 힘들겠지 ? 지금은 괜찮지만 힘들 거예요 .


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[DEVICE] {device}")

[DEVICE] cuda


In [14]:
def scaled_dot_product_attention(query, key, value, mask=None):

    # 1) Q와 K의 내적을 통해 score(유사도) 계산
    # key.transpose(-1, -2): (batch_size, heads, depth, seq_len)
    # matmul 결과 shape: (batch_size, heads, seq_len, seq_len)
    matmul_qk = torch.matmul(query, key.transpose(-1, -2))

    # 2) depth에 따라 정규화
    depth = key.size(-1)  # depth = d_model / heads
    logits = matmul_qk / math.sqrt(depth)

    # 3) 마스크가 주어졌다면 -1e9(아주 작은 값)를 더해 소프트맥스에서 제외시키도록 함
    if mask is not None:
        logits = logits + (mask * -1e9)

    # 4) 소프트맥스 계산해 attention weights 생성
    attention_weights = F.softmax(logits, dim=-1)

    # 5) attention weights와 value의 내적
    output = torch.matmul(attention_weights, value)

    return output, attention_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        # d_model은 num_heads로 나누어떨어져야 함
        assert d_model % num_heads == 0

        self.depth = d_model // num_heads

        # 파이토치에서 Dense는 nn.Linear로 대응
        # 각각 W_Q, W_K, W_V
        self.query_dense = nn.Linear(d_model, d_model)
        self.key_dense = nn.Linear(d_model, d_model)
        self.value_dense = nn.Linear(d_model, d_model)

        self.out_dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        """
        x: (batch_size, seq_len, d_model)
        => (batch_size, num_heads, seq_len, depth) 형태로 변환
        """
        x = x.view(batch_size, -1, self.num_heads, self.depth) # (batch_size, seq_len, num_heads, depth)
        x = x.permute(0, 2, 1, 3)  # (batch_size, num_heads, seq_len, depth)
        return x

    def forward(self, query, key, value, mask=None):
        """
        query, key, value: (batch_size, seq_len, d_model)
        mask: (batch_size, 1, seq_len, seq_len) 등으로 broadcast 가능하도록 구성
        """
        batch_size = query.size(0)

        # Q, K, V에 각각 Linear 적용
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        # Head 분할
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        # 스케일드 닷 프로덕트 어텐션
        scaled_attention, _ = scaled_dot_product_attention(query, key, value, mask)

        # (batch_size, num_heads, seq_len, depth) -> (batch_size, seq_len, num_heads, depth)
        scaled_attention = scaled_attention.permute(0, 2, 1, 3).contiguous()

        # 다시 (batch_size, seq_len, d_model)로 합치기
        concat_attention = scaled_attention.view(batch_size, -1, self.d_model)

        # 최종 Dense
        output = self.out_dense(concat_attention)
        return output

In [15]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super(DecoderBlock, self).__init__()

        # 1. Masked Multi-Head Self-Attention
        self.self_mha = MultiHeadAttention(d_model, num_heads)
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model, eps=1e-6)

        # 2. Feed-Forward Network (Encoder-Decoder Attention은 없음)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(), # 논문에서는 GELU 사용
            nn.Linear(ff_dim, d_model)
        )
        self.dropout2 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model, eps=1e-6)

    def forward(self, x, look_ahead_mask):
        # Masked Multi-Head Self-Attention 수행
        # Query, Key, Value가 모두 자기 자신의 x
        attn_output = self.self_mha(x, x, x, mask=look_ahead_mask)
        attn_output = self.dropout1(attn_output)

        # 잔차 연결(Residual Connection) 및 레이어 정규화(Layer Normalization)
        out1 = self.norm1(x + attn_output)

        # Feed-Forward Network 수행
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)

        # 잔차 연결 및 레이어 정규화
        out2 = self.norm2(out1 + ffn_output)

        return out2

In [53]:
class GPT(nn.Module):
    """
    Decoder-only Transformer 모델 (GPT)
    """
    def __init__(self,
                 vocab_size,
                 num_layers,      # 디코더 블록 층 수
                 ff_dim,          # Feed-forward 네트워크의 중간 차원
                 d_model,         # 임베딩 및 내부 표현 차원
                 num_heads,       # 멀티헤드 어텐션의 헤드 수
                 max_len,         # 최대 시퀀스 길이
                 dropout=0.1):
        super(GPT, self).__init__()
        self.d_model = d_model

        # 1. 입력 임베딩 (토큰 임베딩)
        self.embedding = nn.Embedding(vocab_size, d_model)

        # 2. 위치 임베딩 (단순히 Embedding layer를 추가하면 됨)
        self.pos_embedding = nn.Embedding(max_len, d_model)
        # positional Encoding처럼 max len에 vocab size를 넣으면 안됨

        # nn.Embedding(vocab_size, d_model) -> 단어의 의미 학습
        # nn.Embedding(max_len, d_model) -> 단어의 위치/순서 학습

        self.dropout = nn.Dropout(dropout)

        # 3. DecoderBlock을 num_layers만큼 쌓기
        self.decoder_layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # 4. 최종 출력층 (다음 단어 예측을 위한 어휘 크기의 출력)
        self.final_linear = nn.Linear(d_model, vocab_size)
        # Finetuning할 때, 이 부분을 교체

    def forward(self, x):
        # 입력 x에 대한 look-ahead 마스크 생성
        look_ahead_mask = create_look_ahead_mask(x)

        # 1. 토큰 임베딩 + 위치 임베딩
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0) # shape: (1, seq_len)

        x = self.embedding(x) * math.sqrt(self.d_model) + self.pos_embedding(positions)
        x = self.dropout(x)

        # 2. 모든 디코더 블록을 순서대로 통과
        for layer in self.decoder_layers:
            x = layer(x, look_ahead_mask)

        # 3. 최종 선형 레이어를 통과시켜 로짓(logits) 반환
        logits = self.final_linear(x)
        return logits


# 모델 하이퍼파라미터 설정 및 생성
# GPT 논문에서는 768차원의 layer 12개의 768*4차원의 ffn을 사용함
# 여기서는 모두 절반에 해당하는 파라미터 사용 (head 수는 논문과 동일하게 12개)
NUM_LAYERS = 6
D_MODEL = 384
NUM_HEADS = 12
UNITS = D_MODEL * 4 # ff_dim
DROPOUT = 0.1
MAX_LENGTH = 40 # GPTDataset에서 설정한 max_length와 동일해야 함

model = GPT(
    vocab_size=VOCAB_SIZE,
    num_layers=NUM_LAYERS,
    ff_dim=UNITS,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    max_len=MAX_LENGTH,
    dropout=DROPOUT
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n모델의 총 학습 가능한 파라미터 수: {total_params:,}")


# 간단한 테스트
dummy_input = torch.randint(0, VOCAB_SIZE, (16, MAX_LENGTH - 1)).to(device)
output = model(dummy_input)
print("최종 출력 텐서의 크기:", output.size()) # (배치 크기, 시퀀스 길이, 어휘 크기)

GPT(
  (embedding): Embedding(8000, 384)
  (pos_embedding): Embedding(40, 384)
  (dropout): Dropout(p=0.1, inplace=False)
  (decoder_layers): ModuleList(
    (0-5): 6 x DecoderBlock(
      (self_mha): MultiHeadAttention(
        (query_dense): Linear(in_features=384, out_features=384, bias=True)
        (key_dense): Linear(in_features=384, out_features=384, bias=True)
        (value_dense): Linear(in_features=384, out_features=384, bias=True)
        (out_dense): Linear(in_features=384, out_features=384, bias=True)
      )
      (dropout1): Dropout(p=0.1, inplace=False)
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (ffn): Sequential(
        (0): Linear(in_features=384, out_features=1536, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1536, out_features=384, bias=True)
      )
      (dropout2): Dropout(p=0.1, inplace=False)
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
    )
  )
  (final_linear): Linear(in_features=384

In [39]:
loss_function = nn.CrossEntropyLoss(ignore_index=sp.pad_id())
# 논문에서는 Adam에 L2 정규화를 사용함(lr=2.5e-4, weight_decay=1e-2)
optimizer = optim.AdamW(model.parameters(), lr=2.5e-4, weight_decay = 1e-2)
# 또한 논문에서는 스케줄러를 사용했지만, 여기서는 사용하지 않음.

def accuracy_function(y_pred, y_true, pad_id=0):
    """
    y_pred: (batch_size, seq_len, vocab_size)
    y_true: (batch_size, seq_len)
    """
    preds = y_pred.argmax(dim=-1)  # (batch_size, seq_len)
    mask = (y_true != pad_id)
    correct = (preds == y_true) & mask
    acc = correct.float().sum() / mask.float().sum()
    return acc

In [40]:
def train_step(model, batch, optimizer, loss_function, device):
    """
    GPT 모델을 위한 단일 학습 스텝(step) 함수
    """
    model.train()
    # [변경] 배치 데이터를 모델 입력과 정답 레이블로 분리
    # 이전: enc_input, dec_input, target
    # 변경: model_input, target_label
    model_input, target_label = [x.to(device) for x in batch]

    optimizer.zero_grad()

    # [변경] 모델은 이제 단일 입력만 받습니다.
    logits = model(model_input)  # (batch_size, seq_len, vocab_size)

    # Loss 계산 (CrossEntropyLoss는 (N, C, ...) 형태의 입력을 기대하므로 permute 필요)
    # N: 배치 크기, C: 클래스(어휘) 수, ...: 추가 차원(시퀀스 길이)
    loss = loss_function(logits.permute(0, 2, 1), target_label)

    # 역전파(Backpropagation)
    loss.backward()
    optimizer.step()

    # 정확도 계산 및 손실 반환
    acc = accuracy_function(logits, target_label, pad_id=sp.pad_id())
    return loss.item(), acc

def train(model, dataloader, optimizer, loss_function, num_epochs, device):
    model.to(device)

    for epoch in range(num_epochs):
        total_loss, total_acc = 0, 0
        for step, batch in enumerate(dataloader):
            loss, acc = train_step(model, batch, optimizer, loss_function, device)
            total_loss += loss
            total_acc += acc

        avg_loss = total_loss / len(dataloader)
        avg_acc = total_acc / len(dataloader)
        print(f"Epoch {epoch+1} Completed - Avg Loss: {avg_loss:.4f}, Avg Acc: {avg_acc:.4f}")

In [41]:
train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_function=loss_function,
    num_epochs=30,
    device=device
)

Epoch 1 Completed - Avg Loss: 5.9009, Avg Acc: 0.2331
Epoch 2 Completed - Avg Loss: 4.9734, Avg Acc: 0.2949
Epoch 3 Completed - Avg Loss: 4.4961, Avg Acc: 0.3264
Epoch 4 Completed - Avg Loss: 4.1201, Avg Acc: 0.3528
Epoch 5 Completed - Avg Loss: 3.7856, Avg Acc: 0.3798
Epoch 6 Completed - Avg Loss: 3.4698, Avg Acc: 0.4099
Epoch 7 Completed - Avg Loss: 3.1680, Avg Acc: 0.4427
Epoch 8 Completed - Avg Loss: 2.8705, Avg Acc: 0.4793
Epoch 9 Completed - Avg Loss: 2.5930, Avg Acc: 0.5194
Epoch 10 Completed - Avg Loss: 2.3257, Avg Acc: 0.5599
Epoch 11 Completed - Avg Loss: 2.0801, Avg Acc: 0.6020
Epoch 12 Completed - Avg Loss: 1.8644, Avg Acc: 0.6412
Epoch 13 Completed - Avg Loss: 1.6641, Avg Acc: 0.6802
Epoch 14 Completed - Avg Loss: 1.4956, Avg Acc: 0.7135
Epoch 15 Completed - Avg Loss: 1.3498, Avg Acc: 0.7441
Epoch 16 Completed - Avg Loss: 1.2351, Avg Acc: 0.7694
Epoch 17 Completed - Avg Loss: 1.1421, Avg Acc: 0.7883
Epoch 18 Completed - Avg Loss: 1.0672, Avg Acc: 0.8035
Epoch 19 Completed 

In [42]:
def gpt_inference(model, sentence, tokenizer, device='cpu', max_length=40):
    """
    GPT 모델을 위한 추론(생성) 함수
    """
    model.eval()  # 모델을 평가 모드로 설정

    # 특수 토큰 ID 가져오기
    bos_id = tokenizer.bos_id()
    eos_id = tokenizer.eos_id()

    # 1. 입력 문장 전처리 및 토큰화
    sentence = normalize_text(sentence)
    token_ids = tokenizer.encode(sentence)

    # 2. 모델의 초기 입력 시퀀스 준비
    # GPT는 입력 문장을 프롬프트로 사용하여 다음을 예측.
    # 학습 데이터 형식([BOS] Q [EOS] A [EOS])과 유사하게
    # [BOS] + 입력문장_토큰 + [EOS] 형태로 초기 입력을 구성.
    # 이는 모델에게 "이제 답변을 생성할 차례"라는 신호를 줌.
    model_input = torch.tensor([[bos_id] + token_ids + [eos_id]], dtype=torch.long, device=device)

    # torch.no_grad() 컨텍스트 내에서 그래디언트 계산을 비활성화하여 메모리 사용량을 줄이고 계산 속도를 높임
    with torch.no_grad():
        # 3. 한 토큰씩 자동회귀적(Autoregressive)으로 생성
        for _ in range(max_length):
            # 모델에 현재까지 생성된 전체 시퀀스를 입력으로 전달
            logits = model(model_input)  # shape: (1, current_seq_len, vocab_size)

            # 마지막 토큰의 로짓(logits)만 추출하여 다음 토큰 예측에 사용
            last_token_logits = logits[:, -1, :]  # shape: (1, vocab_size)
            predicted_id = torch.argmax(last_token_logits, dim=-1) # shape: (1,)

            # 4. 생성된 토큰이 종료(EOS) 토큰이면 생성을 중단
            if predicted_id.item() == eos_id:
                break

            # 5. 예측된 토큰을 현재 시퀀스에 이어붙여 다음 스텝의 입력으로 사용
            model_input = torch.cat([model_input, predicted_id.unsqueeze(0)], dim=1)

    # 최종 생성된 토큰 ID 시퀀스를 리스트로 변환하여 반환
    output_ids = model_input.squeeze(0).tolist()
    return output_ids


def sentence_generation(model, sentence, tokenizer, device='cpu'):
    """
    gpt_inference를 호출하여 문장을 생성하고 결과를 출력하는 래퍼(wrapper) 함수
    """
    # gpt_inference 함수로 전체 시퀀스(입력+생성) 생성
    output_seq = gpt_inference(model, sentence, tokenizer, device=device)

    # 생성된 ID 시퀀스에서 프롬프트 부분을 제외하고 순수 답변 부분만 추출
    prompt_ids = tokenizer.encode(sentence)
    # [BOS] + 프롬프트 ID + [EOS] 이후의 부분이 생성된 답변에 해당
    # (len(prompt_ids) + 2) -> BOS, EOS 두 개 토큰
    start_index = len(prompt_ids) + 2
    predicted_ids = output_seq[start_index:]

    # 만약 생성된 토큰 중 EOS가 있다면 그 전까지만 사용
    if tokenizer.eos_id() in predicted_ids:
        eos_index = predicted_ids.index(tokenizer.eos_id())
        predicted_ids = predicted_ids[:eos_index]

    # 최종 답변을 디코딩하여 문장으로 복원
    predicted_sentence = tokenizer.decode(predicted_ids)

    print("\n--- 생성 결과 ---")
    print("입력 문장:", sentence)
    print("생성 답변:", predicted_sentence)
    return predicted_sentence

In [51]:
sentences_to_test = [
    '나 오늘 너무 화가나.',
    '안녕',
    '밥 먹었어?',
    '오늘 좀 우울해.',
    '엄마',
    '아빠',
]

# 리스트를 순회하며 각 문장에 대한 답변을 생성
for sentence in sentences_to_test:
    sentence_generation(model, sentence, sp, device)
    print("="*50)


--- 생성 결과 ---
입력 문장: 나 오늘 너무 화가나.
생성 답변: 자신에게 더 좋은 날이네요 .

--- 생성 결과 ---
입력 문장: 안녕
생성 답변: 안녕하세요 .

--- 생성 결과 ---
입력 문장: 밥 먹었어?
생성 답변: 저는 배터리가 밥이예요 .

--- 생성 결과 ---
입력 문장: 오늘 좀 우울해.
생성 답변: 제 앞에서 울어도 돼요 .

--- 생성 결과 ---
입력 문장: 엄마
생성 답변: 후회는 후회를 낳을뿐이에요 . 용기 내세요 .

--- 생성 결과 ---
입력 문장: 아빠
생성 답변: 담배 피지 마세요 .
